# Lecture 10: Uncertainty, Ehrenfest, and the Classical Limit
### Computational Companion — Gabor Limit, Gaussian Wave Packets, and Ehrenfest's Theorem

This notebook verifies every claim in Lecture 10:

1. **Uncertainty principle** — $\Delta x \cdot \Delta p \geq \hbar/2$ for various wave packets
2. **Gaussian saturation** — Gaussians achieve $\Delta x \cdot \Delta p = \hbar/2$ exactly
3. **Free particle evolution** — Gaussian spreading, uncertainty product grows
4. **Ehrenfest's theorem** — $d\langle x\rangle/dt = \langle p\rangle/m$ and $d\langle p\rangle/dt = -\langle V'\rangle$
5. **Classical limit** — $\langle V'(\hat{x})\rangle \approx V'(\langle \hat{x}\rangle)$ for narrow wave packets
6. **Gabor limit analogy** — time-frequency uncertainty mirrors position-momentum

**Convention:** $\hbar = 1$ throughout (except where noted).

**Reference:** Lecture 10 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simps

hbar = 1.0  # natural units

# ── Grid for position-space wave functions ──
N = 4096
L = 40.0  # domain [-L/2, L/2]
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)

# Momentum grid (FFT convention)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N, d=dx) * 2 * np.pi * hbar

def normalize(psi):
    """Normalize a position-space wave function."""
    norm = np.sqrt(np.sum(np.abs(psi)**2) * dx)
    return psi / norm

def expect_x(psi):
    """⟨x⟩ for a position-space wave function."""
    prob = np.abs(psi)**2 * dx
    return np.sum(x * prob)

def expect_x2(psi):
    return np.sum(x**2 * np.abs(psi)**2 * dx)

def delta_x(psi):
    return np.sqrt(expect_x2(psi) - expect_x(psi)**2)

def expect_p(psi):
    """⟨p⟩ via FFT."""
    psi_p = np.fft.fft(psi) * dx / np.sqrt(2 * np.pi * hbar)
    prob_p = np.abs(psi_p)**2 * dp
    return np.sum(p * prob_p)

def expect_p2(psi):
    psi_p = np.fft.fft(psi) * dx / np.sqrt(2 * np.pi * hbar)
    prob_p = np.abs(psi_p)**2 * dp
    return np.sum(p**2 * prob_p)

def delta_p(psi):
    return np.sqrt(expect_p2(psi) - expect_p(psi)**2)

def evolve_free(psi, t, m=1.0):
    """Free-particle evolution via split-step FFT."""
    psi_p = np.fft.fft(psi)
    phase = np.exp(-1j * p**2 * t / (2 * m * hbar))
    return np.fft.ifft(psi_p * phase)

print(f"Grid: N={N}, L={L}, dx={dx:.4f}, dp={dp:.4f}")
print(f"ℏ = {hbar}")
print("Setup complete.")

## 1. Uncertainty Principle — Various Wave Packets

We verify $\Delta x \cdot \Delta p \geq \hbar/2$ for several different wave packet shapes. The Gaussian should saturate the bound exactly.

In [ ]:
# ── 1. Uncertainty Principle ──

print("=" * 65)
print("UNCERTAINTY PRINCIPLE: Δx·Δp ≥ ℏ/2 for various wave packets")
print("=" * 65)

sigma = 1.0

wave_packets = {
    "Gaussian (σ=1)": normalize(np.exp(-x**2 / (2 * sigma**2))),
    "Gaussian (σ=0.5)": normalize(np.exp(-x**2 / (2 * 0.5**2))),
    "Gaussian (σ=2)": normalize(np.exp(-x**2 / (2 * 2.0**2))),
    "Rect (width=2)": normalize(np.where(np.abs(x) < 1, 1.0, 0.0)),
    "Triangle": normalize(np.maximum(0, 1 - np.abs(x))),
    "Double Gaussian": normalize(np.exp(-(x-2)**2/2) + np.exp(-(x+2)**2/2)),
    "Lorentzian-like": normalize(1 / (1 + x**2)),
}

print(f"\n  {'Wave packet':<25} {'Δx':>8} {'Δp':>8} {'Δx·Δp':>10} {'≥ ℏ/2':>8} {'Ratio':>8}")
print("  " + "-" * 75)
for name, psi in wave_packets.items():
    dx_val = delta_x(psi)
    dp_val = delta_p(psi)
    product = dx_val * dp_val
    bound = hbar / 2
    print(f"  {name:<25} {dx_val:8.4f} {dp_val:8.4f} {product:10.6f} {product >= bound - 1e-6:>8} {product/bound:8.4f}")

print(f"\n  ℏ/2 = {hbar/2}")
print(f"  Gaussian achieves Δx·Δp = ℏ/2 exactly (ratio ≈ 1.000)")
print(f"  All other shapes have ratio > 1")

## 2. Gaussian Saturation — Minimum Uncertainty State

The Gaussian $v(x) \propto e^{-x^2/(4\sigma^2)}$ is the **unique** minimum-uncertainty state. We verify $\Delta x \cdot \Delta p = \hbar/2$ for many values of $\sigma$, and show that displacing or boosting the Gaussian doesn't change the product.

In [ ]:
# ── 2. Gaussian Saturation ──

print("=" * 65)
print("GAUSSIAN SATURATION: Δx·Δp = ℏ/2 for all widths")
print("=" * 65)

sigmas = np.linspace(0.3, 5.0, 50)
products = []
dx_vals = []
dp_vals = []

for s in sigmas:
    psi = normalize(np.exp(-x**2 / (4 * s**2)))
    dx_v = delta_x(psi)
    dp_v = delta_p(psi)
    dx_vals.append(dx_v)
    dp_vals.append(dp_v)
    products.append(dx_v * dp_v)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(sigmas, dx_vals, 'steelblue', lw=2, label='Δx')
axes[0].plot(sigmas, dp_vals, 'crimson', lw=2, label='Δp')
axes[0].set_xlabel('σ'); axes[0].set_ylabel('Uncertainty')
axes[0].set_title('Δx and Δp vs Gaussian width σ')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sigmas, products, 'green', lw=2)
axes[1].axhline(hbar/2, color='red', ls='--', lw=2, label=f'ℏ/2 = {hbar/2}')
axes[1].set_xlabel('σ'); axes[1].set_ylabel('Δx·Δp')
axes[1].set_title('Δx·Δp = ℏ/2 for all σ (Gaussian saturation)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.5*hbar)

axes[2].plot(dx_vals, dp_vals, 'darkorange', lw=2)
dxr = np.linspace(0.2, 4, 200)
axes[2].plot(dxr, hbar/(2*dxr), 'r--', lw=1.5, label='Δp = ℏ/(2Δx)')
axes[2].set_xlabel('Δx'); axes[2].set_ylabel('Δp')
axes[2].set_title('Δx vs Δp: Gaussians lie on the bound')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Gaussians are minimum-uncertainty states', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Max |Δx·Δp − ℏ/2| over all σ: {np.max(np.abs(np.array(products) - hbar/2)):.6e}")

# Displaced and boosted Gaussian
print("\nDisplaced/boosted Gaussians:")
for x0, p0, label in [(3.0, 0, "x₀=3"), (0, 2.0, "p₀=2"), (3.0, 2.0, "x₀=3, p₀=2")]:
    psi = normalize(np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * p0 * x / hbar))
    print(f"  {label}: Δx·Δp = {delta_x(psi) * delta_p(psi):.6f} (= ℏ/2 = {hbar/2})")

## 3. Free Particle Evolution — Gaussian Spreading

A Gaussian wave packet evolving under the free-particle Hamiltonian $H = p^2/(2m)$ spreads in position while maintaining $\Delta p = \text{const}$. The uncertainty product $\Delta x \cdot \Delta p$ grows beyond $\hbar/2$ for $t > 0$.

In [ ]:
# ── 3. Free Particle Evolution ──

print("=" * 65)
print("FREE PARTICLE: Gaussian spreading")
print("=" * 65)

m = 1.0
sigma0 = 1.0
p0 = 3.0  # initial momentum

psi0 = normalize(np.exp(-x**2 / (4 * sigma0**2)) * np.exp(1j * p0 * x / hbar))

times = np.linspace(0, 8, 200)
dx_t = np.zeros(len(times))
dp_t = np.zeros(len(times))
ex_t = np.zeros(len(times))
ep_t = np.zeros(len(times))

for i, t in enumerate(times):
    psi_t = evolve_free(psi0, t, m)
    dx_t[i] = delta_x(psi_t)
    dp_t[i] = delta_p(psi_t)
    ex_t[i] = expect_x(psi_t)
    ep_t[i] = expect_p(psi_t)

# Analytic spreading: Δx(t) = σ₀ √(1 + (ℏt/(2mσ₀²))²)
dx_analytic = sigma0 * np.sqrt(1 + (hbar * times / (2 * m * sigma0**2))**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Δx(t)
axes[0, 0].plot(times, dx_t, 'steelblue', lw=2, label='Δx (numerical)')
axes[0, 0].plot(times, dx_analytic, 'r--', lw=1.5, label='Δx (analytic)')
axes[0, 0].set_xlabel('t'); axes[0, 0].set_ylabel('Δx')
axes[0, 0].set_title('Position uncertainty spreads')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# Δp(t)
axes[0, 1].plot(times, dp_t, 'crimson', lw=2, label='Δp (numerical)')
axes[0, 1].axhline(dp_t[0], color='green', ls='--', lw=1.5, label=f'Δp(0) = {dp_t[0]:.4f}')
axes[0, 1].set_xlabel('t'); axes[0, 1].set_ylabel('Δp')
axes[0, 1].set_title('Momentum uncertainty is constant')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# Δx·Δp
product_t = dx_t * dp_t
axes[1, 0].plot(times, product_t, 'green', lw=2, label='Δx·Δp')
axes[1, 0].axhline(hbar/2, color='red', ls='--', lw=2, label=f'ℏ/2 = {hbar/2}')
axes[1, 0].set_xlabel('t'); axes[1, 0].set_ylabel('Δx·Δp')
axes[1, 0].set_title('Uncertainty product grows (saturates only at t=0)')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# ⟨x⟩(t) vs classical trajectory
axes[1, 1].plot(times, ex_t, 'steelblue', lw=2, label='⟨x⟩(t) (quantum)')
axes[1, 1].plot(times, p0/m * times, 'r--', lw=1.5, label=f'x = p₀t/m (classical)')
axes[1, 1].set_xlabel('t'); axes[1, 1].set_ylabel('⟨x⟩')
axes[1, 1].set_title('Ehrenfest: ⟨x⟩ follows classical trajectory')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'Free particle: Gaussian wave packet (σ₀={sigma0}, p₀={p0}, m={m})', fontsize=13)
plt.tight_layout(); plt.show()

# Snapshots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
snap_times = [0, 2, 4, 8]
for ax, t in zip(axes, snap_times):
    psi_t = evolve_free(psi0, t, m)
    ax.plot(x, np.abs(psi_t)**2, 'steelblue', lw=2)
    ax.axvline(p0/m * t, color='red', ls='--', alpha=0.7, label=f'x_cl = {p0/m*t:.1f}')
    ax.set_xlim(-5, 35); ax.set_ylim(0, 0.5)
    ax.set_title(f't = {t}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('|ψ(x,t)|² snapshots: wave packet moves and spreads', fontsize=13)
plt.tight_layout(); plt.show()

print(f"⟨p⟩ is constant: max|⟨p⟩ − p₀| = {np.max(np.abs(ep_t - p0)):.2e}")
print(f"Δp is constant: max|Δp − Δp(0)| = {np.max(np.abs(dp_t - dp_t[0])):.2e}")
print(f"At t=0: Δx·Δp = {product_t[0]:.6f} (= ℏ/2)")
print(f"At t=8: Δx·Δp = {product_t[-1]:.4f} (> ℏ/2, spreading)")

## 4. Ehrenfest's Theorem — Particle in a Potential

We verify Ehrenfest's equations numerically for a particle in a harmonic potential $V(x) = \frac{1}{2}m\omega^2 x^2$:
- $d\langle x\rangle/dt = \langle p\rangle/m$
- $d\langle p\rangle/dt = -\langle V'(x)\rangle = -m\omega^2\langle x\rangle$

For the harmonic oscillator, Ehrenfest gives **exact** classical equations (quadratic potential → no Taylor correction).

In [ ]:
# ── 4. Ehrenfest's Theorem ──

print("=" * 65)
print("EHRENFEST: particle in harmonic potential V = ½mω²x²")
print("=" * 65)

m = 1.0
omega_ho = 1.0
V = 0.5 * m * omega_ho**2 * x**2
dVdx = m * omega_ho**2 * x  # V'(x) = mω²x

# Initial displaced Gaussian
x0_init = 3.0
sigma_ho = 1.0 / np.sqrt(m * omega_ho)  # ground-state width
psi0_ho = normalize(np.exp(-(x - x0_init)**2 / (4 * sigma_ho**2)))

# Evolution via split-step method
dt = 0.01
n_steps = 2000
ts_ho = np.arange(n_steps) * dt
ex_ho = np.zeros(n_steps)
ep_ho = np.zeros(n_steps)
eVp_ho = np.zeros(n_steps)  # ⟨V'(x)⟩

psi = psi0_ho.copy()
for i in range(n_steps):
    ex_ho[i] = expect_x(psi)
    ep_ho[i] = expect_p(psi)
    eVp_ho[i] = np.sum(dVdx * np.abs(psi)**2 * dx)
    
    # Split-step: half V, full T, half V
    psi = psi * np.exp(-1j * V * dt / (2 * hbar))
    psi_p = np.fft.fft(psi)
    psi_p = psi_p * np.exp(-1j * p**2 * dt / (2 * m * hbar))
    psi = np.fft.ifft(psi_p)
    psi = psi * np.exp(-1j * V * dt / (2 * hbar))

# Numerical derivatives
dxdt_num = np.gradient(ex_ho, dt)
dpdt_num = np.gradient(ep_ho, dt)

# Ehrenfest predictions
dxdt_ehr = ep_ho / m
dpdt_ehr = -eVp_ho

# Classical solution
x_classical = x0_init * np.cos(omega_ho * ts_ho)
p_classical = -m * omega_ho * x0_init * np.sin(omega_ho * ts_ho)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(ts_ho, ex_ho, 'steelblue', lw=2, label='⟨x⟩ (quantum)')
axes[0, 0].plot(ts_ho, x_classical, 'r--', lw=1.5, label='x(t) (classical)')
axes[0, 0].set_xlabel('t'); axes[0, 0].set_ylabel('⟨x⟩')
axes[0, 0].set_title('⟨x⟩(t) matches classical trajectory exactly')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(ts_ho, ep_ho, 'crimson', lw=2, label='⟨p⟩ (quantum)')
axes[0, 1].plot(ts_ho, p_classical, 'r--', lw=1.5, label='p(t) (classical)')
axes[0, 1].set_xlabel('t'); axes[0, 1].set_ylabel('⟨p⟩')
axes[0, 1].set_title('⟨p⟩(t) matches classical trajectory exactly')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(ts_ho, dxdt_num, 'steelblue', lw=2, label='d⟨x⟩/dt (numerical)')
axes[1, 0].plot(ts_ho, dxdt_ehr, 'r--', lw=1.5, label='⟨p⟩/m (Ehrenfest)')
axes[1, 0].set_xlabel('t'); axes[1, 0].set_ylabel('d⟨x⟩/dt')
axes[1, 0].set_title('Velocity equation: d⟨x⟩/dt = ⟨p⟩/m')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ts_ho, dpdt_num, 'crimson', lw=2, label='d⟨p⟩/dt (numerical)')
axes[1, 1].plot(ts_ho, dpdt_ehr, 'r--', lw=1.5, label='−⟨V\'(x)⟩ (Ehrenfest)')
axes[1, 1].set_xlabel('t'); axes[1, 1].set_ylabel('d⟨p⟩/dt')
axes[1, 1].set_title('Force equation: d⟨p⟩/dt = −⟨V\'(x)⟩')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'Ehrenfest: harmonic oscillator (ω={omega_ho}, x₀={x0_init})', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Max |⟨x⟩ − x_classical|: {np.max(np.abs(ex_ho - x_classical)):.6f}")
print(f"Max |d⟨x⟩/dt − ⟨p⟩/m|: {np.max(np.abs(dxdt_num - dxdt_ehr)[10:-10]):.6f}")
print(f"→ Harmonic oscillator: Ehrenfest gives EXACT classical equations")

## 5. Classical Limit — Narrow vs Wide Wave Packets

For non-quadratic potentials, $\langle V'(\hat{x})\rangle \neq V'(\langle\hat{x}\rangle)$. The deviation is controlled by $(\Delta x)^2 \cdot V'''$. We compare a narrow wave packet (classical-like) vs a wide wave packet in an anharmonic potential $V(x) = x^4$.

In [ ]:
# ── 5. Classical Limit ──

print("=" * 65)
print("CLASSICAL LIMIT: narrow vs wide wave packets in V(x) = x⁴")
print("=" * 65)

V_quartic = 0.1 * x**4
dV_quartic = 0.4 * x**3

results = {}
for label, sigma_cl in [("Narrow (σ=0.3)", 0.3), ("Wide (σ=2.0)", 2.0)]:
    x0_cl = 3.0
    psi_cl = normalize(np.exp(-(x - x0_cl)**2 / (4 * sigma_cl**2)))
    
    dt_cl = 0.005
    n_steps_cl = 1000
    ts_cl = np.arange(n_steps_cl) * dt_cl
    ex_cl = np.zeros(n_steps_cl)
    ep_cl = np.zeros(n_steps_cl)
    eVp_cl = np.zeros(n_steps_cl)  # ⟨V'(x)⟩
    Vp_of_ex = np.zeros(n_steps_cl)  # V'(⟨x⟩)
    
    psi = psi_cl.copy()
    for i in range(n_steps_cl):
        ex_cl[i] = expect_x(psi)
        ep_cl[i] = expect_p(psi)
        eVp_cl[i] = np.sum(dV_quartic * np.abs(psi)**2 * dx)
        Vp_of_ex[i] = 0.4 * ex_cl[i]**3
        
        psi = psi * np.exp(-1j * V_quartic * dt_cl / (2 * hbar))
        psi_p = np.fft.fft(psi)
        psi_p = psi_p * np.exp(-1j * p**2 * dt_cl / (2 * m * hbar))
        psi = np.fft.ifft(psi_p)
        psi = psi * np.exp(-1j * V_quartic * dt_cl / (2 * hbar))
    
    results[label] = (ts_cl, ex_cl, ep_cl, eVp_cl, Vp_of_ex)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (label, (ts, ex, ep, eVp, Vp_ex)) in enumerate(results.items()):
    row = idx
    axes[row, 0].plot(ts, ex, 'steelblue', lw=2, label='⟨x⟩ (quantum)')
    axes[row, 0].set_xlabel('t'); axes[row, 0].set_ylabel('⟨x⟩')
    axes[row, 0].set_title(f'{label}: ⟨x⟩(t)'); axes[row, 0].grid(alpha=0.3); axes[row, 0].legend()
    
    axes[row, 1].plot(ts, eVp, 'crimson', lw=2, label='⟨V\'(x)⟩')
    axes[row, 1].plot(ts, Vp_ex, 'green', lw=2, ls='--', label='V\'(⟨x⟩)')
    axes[row, 1].set_xlabel('t'); axes[row, 1].set_ylabel('Force')
    axes[row, 1].set_title(f'{label}: ⟨V\'(x)⟩ vs V\'(⟨x⟩)')
    axes[row, 1].legend(); axes[row, 1].grid(alpha=0.3)

plt.suptitle('Classical limit: ⟨V\'(x̂)⟩ ≈ V\'(⟨x̂⟩) only for narrow wave packets', fontsize=13)
plt.tight_layout(); plt.show()

for label, (ts, ex, ep, eVp, Vp_ex) in results.items():
    err = np.max(np.abs(eVp - Vp_ex))
    print(f"  {label}: max|⟨V'(x)⟩ − V'(⟨x⟩)| = {err:.4f}")
print(f"\n→ Narrow wave packet: ⟨V'⟩ ≈ V'(⟨x⟩) (classical-like)")
print(f"  Wide wave packet: significant deviation (quantum effects)")

## 6. Gabor Limit Analogy — Time-Frequency Uncertainty

The quantum uncertainty principle $\Delta x \cdot \Delta p \geq \hbar/2$ is the same mathematical theorem as the Gabor limit $\Delta t \cdot \Delta\omega \geq 1/2$. We demonstrate the Gabor limit with signals and show the exact parallel.

In [ ]:
# ── 6. Gabor Limit Analogy ──

print("=" * 65)
print("GABOR LIMIT: Δt·Δω ≥ 1/2 (same theorem as Δx·Δp ≥ ℏ/2)")
print("=" * 65)

# Time-domain grid
N_sig = 4096
T_sig = 20.0
dt_sig = T_sig / N_sig
t_sig = np.linspace(-T_sig/2, T_sig/2, N_sig, endpoint=False)
omega_sig = np.fft.fftfreq(N_sig, d=dt_sig) * 2 * np.pi

def normalize_sig(s):
    return s / np.sqrt(np.sum(np.abs(s)**2) * dt_sig)

def delta_t_sig(s):
    prob = np.abs(s)**2 * dt_sig
    et = np.sum(t_sig * prob)
    et2 = np.sum(t_sig**2 * prob)
    return np.sqrt(et2 - et**2)

def delta_omega_sig(s):
    S = np.fft.fft(s) * dt_sig
    prob_w = np.abs(S)**2 * (omega_sig[1] - omega_sig[0]) / (2 * np.pi)
    prob_w /= np.sum(prob_w)  # normalize
    ew = np.sum(omega_sig * prob_w)
    ew2 = np.sum(omega_sig**2 * prob_w)
    return np.sqrt(ew2 - ew**2)

signals = {
    "Gaussian (σ=0.5)": normalize_sig(np.exp(-t_sig**2 / (2 * 0.5**2))),
    "Gaussian (σ=1.0)": normalize_sig(np.exp(-t_sig**2 / (2 * 1.0**2))),
    "Gaussian (σ=2.0)": normalize_sig(np.exp(-t_sig**2 / (2 * 2.0**2))),
    "Rect (width=2)": normalize_sig(np.where(np.abs(t_sig) < 1, 1.0, 0.0)),
    "Triangle": normalize_sig(np.maximum(0, 1 - np.abs(t_sig))),
}

print(f"\n  {'Signal':<25} {'Δt':>8} {'Δω':>8} {'Δt·Δω':>10} {'≥ 1/2':>8} {'Ratio':>8}")
print("  " + "-" * 75)
for name, s in signals.items():
    dt_v = delta_t_sig(s)
    dw_v = delta_omega_sig(s)
    prod = dt_v * dw_v
    print(f"  {name:<25} {dt_v:8.4f} {dw_v:8.4f} {prod:10.6f} {prod >= 0.5 - 1e-4:>8} {prod/0.5:8.4f}")

print(f"\n  Gabor limit: Δt·Δω ≥ 1/2")
print(f"  Quantum:     Δx·Δp ≥ ℏ/2")
print(f"  Same theorem. Same Gaussians saturate both. Same Fourier duality.")